# Re-compute Cross-match with Period Matching

**Problem**: The previous `match_source='confirmed'` only checked whether the **host star name** matched NASA's confirmed planets table. A star can host 3 known planets at different periods, but BLS might have picked up noise or a different signal. The old logic would still label it "confirmed".

**Fix**: Label a signal as `已知` only when BOTH conditions are met:
1. The host star name matches a NASA confirmed planet host
2. **The BLS-found `Period (d)` matches at least one confirmed planet's orbital period for that star** (within 5% tolerance)

**Input**: `01.csv` (the CSV with BLS results, containing `Target` and `Period (d)` columns but missing `已知/新候选`)

**Output**: Same CSV with a corrected `已知/新候选` column appended.

## 0. Install

In [ ]:
!pip install -q numpy pandas astropy astroquery

## 1. Load Data

In [ ]:
import warnings
import numpy as np
import pandas as pd
from collections import Counter

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 280)

INPUT_CSV  = '01.csv'
OUTPUT_CSV = '01_corrected.csv'
PERIOD_TOLERANCE = 0.05  # 5% relative tolerance for period matching

df = pd.read_csv(INPUT_CSV)
print(f'Loaded: {len(df)} rows')
print(f'Columns: {list(df.columns)}')
print(f'Has 已知/新候选: {"已知/新候选" in df.columns}')
print(f'Has match_source: {"match_source" in df.columns}')
df.head(3)

## 2. Query NASA Confirmed Planets with Host + Period Matching

For each target: query `pscomppars` by host star name, then check if **any confirmed planet around that star has an orbital period matching the BLS `Period (d)`** within 5%.

In [ ]:
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

_CACHE = {}  # key: (target,) -> list of orbital periods for that star

def _val(x):
    '''Safely extract float from astropy Quantity or masked value.'''
    if x is None: return np.nan
    if np.ma.is_masked(x): return np.nan
    if hasattr(x, 'value'): return float(x.value)
    try: return float(x)
    except: return np.nan

def get_confirmed_periods(target):
    '''Return a list of known orbital periods for confirmed planets around this host star.
    Returns None if the star is not in the confirmed planets table.
    Returns [] if query failed.'''
    t = str(target)
    if t in _CACHE:
        return _CACHE[t]
    try:
        result = NasaExoplanetArchive.query_object(t)
        if result is None or len(result) == 0:
            _CACHE[t] = None   # star not in confirmed table
            return None
        periods = []
        for r in result:
            p = _val(r['pl_orbper'])
            if np.isfinite(p) and p > 0:
                periods.append(round(p, 4))
        _CACHE[t] = periods if periods else None
        return _CACHE[t]
    except Exception:
        _CACHE[t] = []  # query error, don't retry
        return []

def classify_signal(target, bls_period, tic_id=None):
    '''Returns (label, source, matched_period).
    - 已知: host star is confirmed AND BLS period matches a known planet period
    - 新候选: host star not confirmed, OR confirmed but no period match
    - query_error: NASA archive query failed
    '''
    bls_p = float(bls_period)
    known_periods = get_confirmed_periods(target)

    # Query error
    if known_periods is not None and len(known_periods) == 0:
        return '新候选', 'query_error', np.nan

    # Star not in confirmed table -> try TOI by TIC ID
    if known_periods is None:
        # Fallback: check TOI table by TIC ID
        if tic_id is not None and not (isinstance(tic_id, float) and np.isnan(tic_id)):
            try:
                tic_int = int(float(tic_id))
                toi_result = NasaExoplanetArchive.query_criteria(
                    table='toi', select='toi,tid,tfopwg_disp',
                    where=f'tid={tic_int}')
                if toi_result is not None and len(toi_result) > 0:
                    return '已知', 'TOI', np.nan  # TOI match without period check
            except Exception:
                pass
        return '新候选', 'none', np.nan

    # Star is confirmed. Check period match.
    for kp in known_periods:
        if kp > 0 and abs(bls_p - kp) / kp < PERIOD_TOLERANCE:
            return '已知', f'period_match(P_BLS={bls_p:.4f}d, P_NASA={kp:.4f}d)', kp

    # Star is confirmed but no period matches - this BLS signal is NOT the known planet
    return '新候选', f'no_period_match(星已确认,但P_BLS={bls_p:.4f}d不匹配任何已知周期{known_periods[:3]})', np.nan

print('Functions ready.')

## 3. Diagnostic: Test on First 3 Rows

In [ ]:
print('=== Diagnostic (first 3 targets) ===')
for i in range(min(3, len(df))):
    row = df.iloc[i]
    tgt, bls_p, tic = row['Target'], row['Period (d)'], row.get('TIC_ID', None)
    periods = get_confirmed_periods(tgt)
    label, source, matched_p = classify_signal(tgt, bls_p, tic)
    print(f"\n  [{i+1}] {tgt}  P_BLS={bls_p:.4f}d")
    print(f"      NASA known periods: {periods}")
    print(f"      -> {label}  ({source})")
    if matched_p and np.isfinite(matched_p):
        diff = abs(bls_p - matched_p) / matched_p * 100
        print(f"      Match: P_BLS={bls_p:.4f} vs P_NASA={matched_p:.4f}  ({diff:.1f}% off)")

## 4. Batch Re-classify All Rows

In [ ]:
print(f'=== Processing {len(df)} rows ===')

labels, sources, matched_periods = [], [], []
has_tic = 'TIC_ID' in df.columns

for i, row in df.iterrows():
    tgt   = row['Target']
    bls_p = row['Period (d)']
    tic   = row['TIC_ID'] if has_tic else None

    lb, src, mp = classify_signal(tgt, bls_p, tic)
    labels.append(lb)
    sources.append(src)
    matched_periods.append(mp)

    print(f"[{i+1:>3}/{len(df)}] {str(tgt)[:22]:<22} P_BLS={bls_p:.4f}d -> {lb:<6} ({src})")

# Write new columns
df['已知/新候选'] = labels
df['match_source'] = sources
df['NASA_matched_period'] = matched_periods

# Save
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'\n{"="*60}')
print(f'Re-classification complete.')
print(f'  Known (period match):    {labels.count("已知")}')
print(f'  New Candidate:           {labels.count("新候选")}')
print(f'Source distribution: {dict(Counter(sources))}')
print(f'Saved -> {OUTPUT_CSV}')

## 5. Compare with Original `match_source` (if present)

Show which old-confirmed targets lost their status because their BLS period didn't match any known planet period.

In [ ]:
if 'match_source' in df.columns:
    # Check if original match_source was stored with a different name
    # The original match_source column was overwritten above
    pass

# Show targets that are now '新候选' despite being host stars of confirmed planets
no_match = df[(df['已知/新候选'] == '新候选') & (df['match_source'].str.startswith('no_period_match'))]
print(f'Host stars with confirmed planets but no period match: {len(no_match)}')
if len(no_match) > 0:
    print(no_match[['Target', 'Period (d)', 'match_source']].to_string(index=False))

# Final preview
print(f'\nFinal table preview:')
df.head(10)